# Archivus – Drive Management

This notebook covers drive access management after initial setup.

| Step | Endpoint | Method |
|------|----------|--------|
| 1 | `/auth/drive/invite` – generate invite code | POST |
| 2 | `/auth/drive/add` – directly add a user | POST |
| 3 | `/auth/drive/users` – list all users in drive | GET |
| 4 | `/auth/drive/info` – detailed drive view | GET |
| 5 | `/auth/drive/remove` – remove a user | POST |

**Pre-requisites:**
- Admin user `samar` must exist (run `api.ipynb` first, or cells 1-2 here set it up).
- `newuser2` must exist or will be created here.
- Server running on `http://localhost:8080`.

In [ ]:
import requests, json

BASE = 'http://localhost:8080'

def show(resp):
    try:
        body = resp.json()
    except Exception:
        body = resp.text
    print(f'Status : {resp.status_code}')
    print(f'Body   : {json.dumps(body, indent=2)}')
    return body

admin_token  = None
drive_id     = None
invite_code  = None
new_token    = None
new_user_id  = None

## 1 · Login as admin

All drive-management endpoints require an admin token.

In [ ]:
# Register admin if not yet created (safe to run multiple times — duplicate is ignored)
requests.post(f'{BASE}/auth/register', json={
    'username': 'samar', 'password': 'password12', 'pin': '123456',
    'email': 'samar@example.com', 'user_type': 'business', 'is_admin': True,
})

resp = requests.post(f'{BASE}/auth/login', json={'username':'samar','pin':'123456'})
body = show(resp)
admin_token = body['token']

In [ ]:
resp = requests.get(f'{BASE}/auth/user/info',
    headers={'Authorization': f'Bearer {admin_token}'})
body = show(resp)

admin_user_id = body['user']['ID']
drives  = body.get('drives', [])
drive_id = drives[0]['DriveID'] if drives else None
print(f'\nadmin_user_id : {admin_user_id}')
print(f'drive_id      : {drive_id}')

## 2 · Invite a user with read access

Generates a one-time code that a new user can supply at registration to be  
automatically added to the drive with the specified access level.

In [ ]:
resp = requests.post(f'{BASE}/auth/drive/invite',
    headers={'Authorization': f'Bearer {admin_token}'},
    json={'drive_id': drive_id, 'access': 'read'})
body = show(resp)
invite_code = body.get('invite_code')
print(f'\ninvite_code : {invite_code}')

In [ ]:
# Register newuser2 using the invite code
requests.post(f'{BASE}/auth/register', json={
    'username': 'newuser2', 'password': 'newuserpass12', 'pin': '654321',
    'email': 'newuser2@example.com', 'user_type': 'business',
    'is_admin': False, 'invite_code': invite_code,
})

resp = requests.post(f'{BASE}/auth/login', json={'username':'newuser2','pin':'654321'})
body = show(resp)
new_token = body['token']

In [ ]:
resp = requests.get(f'{BASE}/auth/user/info',
    headers={'Authorization': f'Bearer {new_token}'})
body = show(resp)

new_user_id = body['user']['ID']
drives = body.get('drives', [])
drive_ids = [d['DriveID'] for d in drives]
print(f'\nnew_user_id : {new_user_id}')
print(f'drive IDs   : {drive_ids}')
assert drive_id in drive_ids, 'ERROR: invited drive not in new user drive list'
print('✓ invited drive visible to new user')

## 3 · Get drive info

Full breakdown of the drive: owner, slug, and all users grouped by access level.

In [ ]:
resp = requests.get(f'{BASE}/auth/drive/info',
    headers={'Authorization': f'Bearer {admin_token}'},
    params={'drive_id': drive_id})
body = show(resp)

print(f'\nDrive name : {body.get("name")}')
print(f'Slug       : {body.get("slug")}')
print(f'Owner ID   : {body.get("ownerId")}')
print(f'Read users : {[u["Username"] for u in body.get("readUsers", [])]}')
print(f'Write users: {[u["Username"] for u in body.get("writeUsers", [])]}')

## 4 · Invite with write access

Repeat the invite flow but grant `write` access.  
Write users can create folders and upload files.

In [ ]:
resp = requests.post(f'{BASE}/auth/drive/invite',
    headers={'Authorization': f'Bearer {admin_token}'},
    json={'drive_id': drive_id, 'access': 'write'})
body = show(resp)
write_invite_code = body.get('invite_code')
print(f'\nwrite invite_code : {write_invite_code}')

In [ ]:
requests.post(f'{BASE}/auth/register', json={
    'username': 'writeuser', 'password': 'writepass12', 'pin': '111222',
    'email': 'writeuser@example.com', 'user_type': 'business',
    'is_admin': False, 'invite_code': write_invite_code,
})
resp = requests.post(f'{BASE}/auth/login', json={'username':'writeuser','pin':'111222'})
body = show(resp)
write_token = body['token']
print(f'write_token set: {bool(write_token)}')

## 5 · Add a user directly (no invite code)

An admin can add any existing user to a drive without an invite code  
using `/auth/drive/add`. Useful for bulk onboarding.

In [ ]:
# First get new user's ID if not already set
resp = requests.get(f'{BASE}/auth/user/info',
    headers={'Authorization': f'Bearer {new_token}'})
new_user_id = resp.json()['user']['ID']

# Add them as write user directly
resp = requests.post(f'{BASE}/auth/drive/add',
    headers={'Authorization': f'Bearer {admin_token}'},
    json={
        'user_id':      new_user_id,
        'drive_id':     drive_id,
        'access_level': 'write',
    })
show(resp)

## 6 · List all users in owned drives

Returns drives owned by the caller plus each drive's user breakdown.

In [ ]:
resp = requests.get(f'{BASE}/auth/drive/users',
    headers={'Authorization': f'Bearer {admin_token}'})
body = show(resp)
users_data = body.get('users', [])
for entry in users_data:
    drv = entry.get('drive', {})
    print(f'Drive: {drv.get("Name")}')
    print(f'  read    : {[u["Username"] for u in entry.get("readUsers",  [])]}')
    print(f'  write   : {[u["Username"] for u in entry.get("writeUsers", [])]}')
    print(f'  manager : {[u["Username"] for u in entry.get("managerUsers",[])]}')

## 7 · Remove a user from the drive

- **Owner** can remove anyone.
- **Manager** can remove read/write users (not other managers).
- A user cannot remove themselves.

In [ ]:
resp = requests.post(f'{BASE}/auth/drive/remove',
    headers={'Authorization': f'Bearer {admin_token}'},
    json={
        'user_id':  new_user_id,
        'drive_id': drive_id,
    })
show(resp)

In [ ]:
resp = requests.get(f'{BASE}/auth/drive/info',
    headers={'Authorization': f'Bearer {admin_token}'},
    params={'drive_id': drive_id})
body = show(resp)

all_users = (
    [u['Username'] for u in body.get('readUsers',    [])] +
    [u['Username'] for u in body.get('writeUsers',   [])] +
    [u['Username'] for u in body.get('managerUsers', [])]
)
print(f'\nAll users after removal: {all_users}')
assert 'newuser2' not in all_users, 'ERROR: newuser2 still in drive'
print('✓ newuser2 removed from all access levels')